# ML-04 — Search Intelligence Data Contract

This notebook defines and checks a data contract for the FlyRank warehouse. It uses aggregate counts and metrics only; do not print or commit raw rows, client names, URLs, or search queries.

**Plan fixed before evaluation:** March 2026 is the development feature month, April is its outcome month. May 2026 is the sealed test feature month, and June 2026 is its outcome month. June is not used to invent or tune the label.

The warehouse is gated. Put a Hugging Face **READ** token in Colab Secrets as `HF_TOKEN`; never put the token in a notebook cell or commit it.

In [ ]:
%pip -q install duckdb huggingface_hub pandas scikit-learn

In [ ]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Hugging Face READ token (input hidden): ")

con = duckdb.connect()
escaped_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{escaped_token}')")
del escaped_token

REL = "hf://datasets/FlyRank/internship-warehouse"
def month_source(yyyy_mm):
    return f"read_parquet('{REL}/fact_content_daily_performance/month={yyyy_mm}/*.parquet')"

march_src = month_source("2026-03")
schema = con.sql(f"DESCRIBE SELECT * FROM {march_src}").df()
available_columns = set(schema["column_name"])
required_columns = {
    "report_date", "client_hash_id", "content_hash_id",
    "gsc_impressions", "gsc_clicks", "gsc_avg_position",
    "gsc_data_available", "ga4_data_available", "ga4_sessions"
}
missing_columns = sorted(required_columns - available_columns)
if missing_columns:
    raise ValueError("Warehouse schema differs. Missing columns: " + ", ".join(missing_columns))
print("Schema check passed; required columns are present. No raw rows are displayed.")

## 1. Unit of analysis + time window

**Contract.** One row in the daily fact table represents one `report_date × client_hash_id × content_hash_id` combination. The development feature window is **1–31 March 2026**, with April 2026 as the next-month outcome. The sealed test uses **May 2026** features and **June 2026** outcomes.

The primary source is `fact_content_daily_performance`. Pseudonymous IDs are for grouping and joining only. The next-month outcome is never a final model feature.

In [ ]:
DEVELOPMENT_FEATURE_MONTH = "2026-03"
DEVELOPMENT_OUTCOME_MONTH = "2026-04"
SEALED_TEST_FEATURE_MONTH = "2026-05"
SEALED_TEST_OUTCOME_MONTH = "2026-06"
assert (DEVELOPMENT_FEATURE_MONTH, DEVELOPMENT_OUTCOME_MONTH) == ("2026-03", "2026-04")
assert (SEALED_TEST_FEATURE_MONTH, SEALED_TEST_OUTCOME_MONTH) == ("2026-05", "2026-06")
print("Contract windows are set. June is reserved for final evaluation.")

## 2. Fields: feature / label / context / excluded

**Feature candidates (known before the next-month outcome):**
- `prev_month_impressions` — GSC impressions in the feature month; measured when `gsc_data_available IS TRUE`.
- `prev_month_clicks` — GSC clicks in the feature month; measured when GSC is available.
- `prev_month_avg_position` — average positive GSC position in the feature month; missing when no valid position is measured.
- `prev_month_ga4_sessions` — GA4 sessions only on rows where `ga4_data_available IS TRUE`; otherwise missing, not zero.
- `prev_month_ga4_days_available` — count of days with measured GA4 availability; describes coverage at prediction time.

**Label / proxy:** `declined_next_month`, set to 1 when next-month impressions are at least 20% below feature-month impressions; otherwise 0. This is a directional proxy for decline, not a statement about Google's ranking algorithm.

**Context only:** `client_hash_id`, `content_hash_id`, `report_date`, month markers, availability flags, and coverage counts used for checks or interpreting missingness. IDs and dates are not model inputs.

**Excluded from final predictors:** `next_month_impressions` and any next-month metrics (they overlap the outcome period and leak the answer); raw identifiers, URLs, client names, and private queries (not needed and not safe to publish); and product-decision scores or flags. The deliberately leaked feature is used only in the leakage demonstration, then excluded from the honest feature list.

In [ ]:
FEATURE_CANDIDATES = [
    "prev_month_impressions", "prev_month_clicks", "prev_month_avg_position",
    "prev_month_ga4_sessions", "prev_month_ga4_days_available",
]
LABEL_COLUMN = "declined_next_month"
LEAKED_COLUMN = "next_month_impressions"
assert len(FEATURE_CANDIDATES) <= 5 and LEAKED_COLUMN not in FEATURE_CANDIDATES
print("Allowed feature candidates:", FEATURE_CANDIDATES)
print("Label:", LABEL_COLUMN)
print("Leakage-only column excluded from honest features:", LEAKED_COLUMN)

## 3. Verify it with queries (grain, counts, missing values, windows)

These are the three verification queries required by the contract. They use the March 2026 partition to keep checks scoped to the development window. Outputs are aggregate-only; query 1 should return no duplicate grain groups.

### Verification query 1 — test the row grain

In [ ]:
grain_duplicates = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS rows_at_grain
    FROM {march_src}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Duplicate grain groups returned:", len(grain_duplicates))
if grain_duplicates.empty:
    print("Observed: no duplicate report_date × client × content groups in March.")
else:
    print("Observed: duplicate groups exist; investigate before relying on this grain.")
    display(grain_duplicates[["rows_at_grain"]])

### Verification query 2 — row counts, date span, and missingness

In [ ]:
counts_and_missing = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT client_hash_id) AS distinct_clients,
        COUNT(DISTINCT content_hash_id) AS distinct_content_ids,
        MIN(report_date) AS first_report_date,
        MAX(report_date) AS last_report_date,
        COUNT(*) FILTER (WHERE gsc_impressions IS NULL) AS missing_gsc_impressions,
        COUNT(*) FILTER (WHERE gsc_clicks IS NULL) AS missing_gsc_clicks,
        COUNT(*) FILTER (WHERE gsc_avg_position IS NULL OR gsc_avg_position = 0) AS missing_or_no_position,
        COUNT(*) FILTER (WHERE ga4_sessions IS NULL) AS missing_ga4_sessions
    FROM {march_src}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()
display(counts_and_missing)

### Verification query 3 — availability flags (use `IS TRUE`)

In [ ]:
availability_counts = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS gsc_not_true_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS ga4_not_true_rows
    FROM {march_src}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()
display(availability_counts)
print("IS NOT TRUE includes FALSE and NULL; neither is treated as measured availability.")

## 4. Data limits

- **Unbalanced history:** clients start on different dates. March 2026 is a fixed development slice, not proof that every client has equally long history.
- **GSC-only early rows:** GA4 metrics are used only when `ga4_data_available IS TRUE`. A false or NULL flag is not evidence of zero sessions.
- **Window overlap:** next-month impressions define the proxy label. Any metric from that outcome month would leak the answer if used as a feature. The query-level 90-day table also overlaps recent months, so it is not joined here.
- **Selection and measurement limits:** pages without usable GSC measurements cannot support this label in the same way. The proxy measures observed impression decline, not cause, content quality, or Google's ranking algorithm.
- **Decision-support only:** the analysis can describe measured patterns in this data slice; it cannot establish that a content change caused an outcome.

## Feature frame and deliberate leakage check

This step uses four month partitions only. It builds one row per client-content pair for the March→April development pair and the May→June sealed test pair. The target is fixed as a 20% drop in next-month impressions. The leaked experiment deliberately adds next-month impressions, which directly determines the target; its score is shown only to demonstrate leakage. The final honest feature list excludes that column.

In [ ]:
month_union = " UNION ALL ".join(
    f"SELECT * FROM {month_source(m)}"
    for m in ["2026-03", "2026-04", "2026-05", "2026-06"]
)
feature_frame_sql = f"""
WITH daily AS ({month_union}),
monthly AS (
    SELECT
        CAST(date_trunc('month', report_date) AS DATE) AS month_start,
        client_hash_id, content_hash_id,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_impressions END) AS impressions,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_clicks END) AS clicks,
        AVG(CASE WHEN gsc_data_available IS TRUE AND gsc_avg_position > 0
                 THEN gsc_avg_position END) AS avg_position,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions END) AS ga4_sessions,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_days_available,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_days_available
    FROM daily
    GROUP BY 1, 2, 3
),
paired AS (
    SELECT
        p.client_hash_id, p.content_hash_id, p.month_start AS feature_month,
        p.impressions AS prev_month_impressions,
        p.clicks AS prev_month_clicks,
        p.avg_position AS prev_month_avg_position,
        p.ga4_sessions AS prev_month_ga4_sessions,
        p.ga4_days_available AS prev_month_ga4_days_available,
        p.gsc_days_available AS prev_month_gsc_days_available,
        n.impressions AS next_month_impressions,
        n.gsc_days_available AS next_month_gsc_days_available
    FROM monthly p
    JOIN monthly n
      ON n.client_hash_id = p.client_hash_id
     AND n.content_hash_id = p.content_hash_id
     AND n.month_start = p.month_start + INTERVAL '1 month'
    WHERE (p.month_start = DATE '2026-03-01' AND n.month_start = DATE '2026-04-01')
       OR (p.month_start = DATE '2026-05-01' AND n.month_start = DATE '2026-06-01')
)
SELECT *,
       CAST(next_month_impressions < 0.8 * prev_month_impressions AS INTEGER) AS declined_next_month
FROM paired
WHERE prev_month_impressions >= 100
  AND next_month_impressions IS NOT NULL
  AND prev_month_gsc_days_available > 0
  AND next_month_gsc_days_available > 0
"""
feature_frame = con.sql(feature_frame_sql).df()
feature_frame["feature_month"] = pd.to_datetime(feature_frame["feature_month"])
dev = feature_frame[feature_frame["feature_month"].dt.strftime("%Y-%m") == "2026-03"].copy()
test = feature_frame[feature_frame["feature_month"].dt.strftime("%Y-%m") == "2026-05"].copy()
print(f"Development rows (March features → April outcome): {len(dev):,}")
print(f"Sealed test rows (May features → June outcome): {len(test):,}")
print("No content rows or identifiers are displayed.")
print("Observed development label rate:", round(float(dev[LABEL_COLUMN].mean()), 4) if len(dev) else "no eligible rows")
print("Observed sealed-test label rate:", round(float(test[LABEL_COLUMN].mean()), 4) if len(test) else "no eligible rows")

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, precision_score, recall_score

honest_features = [c for c in FEATURE_CANDIDATES if c in dev.columns and dev[c].notna().any()]
assert LEAKED_COLUMN not in honest_features
if len(dev) == 0 or len(test) == 0:
    raise ValueError("No eligible development/test rows. Check warehouse access, schema, and month partitions.")
if dev[LABEL_COLUMN].nunique() < 2:
    raise ValueError("Development slice contains one label class; do not report a model score.")

def fit_and_score(feature_columns, experiment):
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(n_estimators=150, random_state=42, class_weight="balanced", n_jobs=-1)
    )
    model.fit(dev[feature_columns], dev[LABEL_COLUMN])
    pred = model.predict(test[feature_columns])
    return {
        "experiment": experiment,
        "features_used": len(feature_columns),
        "balanced_accuracy": round(float(balanced_accuracy_score(test[LABEL_COLUMN], pred)), 4),
        "precision": round(float(precision_score(test[LABEL_COLUMN], pred, zero_division=0)), 4),
        "recall": round(float(recall_score(test[LABEL_COLUMN], pred, zero_division=0)), 4),
    }

honest_result = fit_and_score(honest_features, "Honest: prior-month features only")
print("Final honest feature list:", honest_features)
display(pd.DataFrame([honest_result]))

In [ ]:
# Deliberate leakage experiment: next_month_impressions directly determines the label.
# This score is diagnostic only and must not be presented as model performance.
leaky_features = honest_features + [LEAKED_COLUMN]
leaky_result = fit_and_score(leaky_features, "LEAKAGE DEMO ONLY: includes next-month impressions")
display(pd.DataFrame([honest_result, leaky_result]))

FINAL_FEATURES = honest_features.copy()
assert LEAKED_COLUMN not in FINAL_FEATURES
print("Retained honest test score (balanced accuracy):", honest_result["balanced_accuracy"])
print("Leakage-only score is not valid evidence of generalization.")
print("Final model inputs:", FINAL_FEATURES)

## Self-check

- [ ] All three verification queries ran; outputs are visible and claims above match them.
- [ ] The feature frame reports row counts and label rates for development and sealed test.
- [ ] The leakage-only feature is absent from `FINAL_FEATURES`; the honest score is retained.
- [ ] Run all cells top-to-bottom in Colab with your own approved read token.
- [ ] Save the executed notebook back to this repository. Do not commit tokens, raw rows, datasets, URLs, client names, or private queries.

**Interpretation rule:** use *observed*, *measured*, *directional*, and *decision-support*. Do not claim to predict Google's algorithm or establish causation.